<a href="https://colab.research.google.com/github/RoihansLab/Machine-Learning-Projects/blob/main/RAG_Gemini_AVPN_IT_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG (Retrieval-Augmented Generation) dengan Gemini & ChromaDB

## Apa itu RAG?

LLM seperti Gemini punya keterbatasan: pengetahuannya hanya sampai tanggal training-nya.
Mereka tidak tahu tentang:
- Dokumen internal yang belum pernah dipublikasikan
- Data terbaru setelah cutoff date
- Informasi spesifik yang tidak ada di internet

**RAG** adalah teknik yang menggabungkan kemampuan LLM dengan pencarian dokumen.
Alih-alih mengandalkan "memori" model, kita ambil informasi yang relevan dari database dulu,
lalu dikirim ke model sebagai konteks tambahan.

### Arsitektur RAG di notebook ini

```
Dokumen PDF (Pearson VUE AI Certification)
         ↓
    Document Loader      ← baca halaman per halaman
         ↓
      Chunking           ← potong jadi bagian kecil
         ↓
      Embedding          ← ubah teks jadi vector angka
         ↓
     ChromaDB            ← simpan vector di database
         ↓
  User tanya sesuatu
         ↓
      Retriever          ← cari chunk paling relevan
         ↓
    Prompt Template      ← gabungkan konteks + pertanyaan
         ↓
   Gemini (LLM)          ← generate jawaban
         ↓
      Jawaban
```

### Dokumen yang digunakan

Notebook ini menggunakan dokumen **IT Specialist: Artificial Intelligence** dari **Pearson VUE** —
sebuah exam objective guide yang menjelaskan topik-topik yang diujikan dalam sertifikasi AI mereka.
Kita akan membangun RAG yang bisa menjawab pertanyaan tentang isi dokumen ini.


## Instalasi Library

Package yang kita butuhkan:
- `google-generativeai` — SDK resmi Google untuk Gemini API
- `langchain` & `langchain-google-genai` — framework RAG + integrasi Gemini
- `langchain_community` — berisi connector ke ChromaDB dan document loaders
- `pypdf` — membaca file PDF
- `chromadb` — vector database yang akan kita gunakan untuk menyimpan embedding


In [ ]:
!pip install -q -U google-generativeai

In [ ]:
!pip install -U -q google-generativeai langchain langchain-google-genai langchain_community pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from IPython.display import Markdown as md

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata

## Konfigurasi API Key & Model

Kita perlu dua hal sebelum mulai:

### 1. Google AI API Key (untuk Gemini & Embedding)

Cara mendapatkan:
1. Buka [https://aistudio.google.com](https://aistudio.google.com)
2. Klik **Get API Key** → **Create API Key**
3. Di Google Colab: klik ikon 🔑 (Secrets) di sidebar kiri → tambahkan secret `GEMINI`

### 2. Model yang digunakan

- **Gemini 2.5 Flash Lite** — model LLM ringan dari Google, digunakan untuk generate jawaban akhir
- **gemini-embedding-2** — model embedding dari Google, mengubah teks menjadi vector

> Model dan API key cukup dikonfigurasi sekali di sini dan akan dipakai di seluruh notebook.


In [ ]:
import google.generativeai as genai

GEMINI = userdata.get('GEMINI')
api_key = GEMINI
genai.configure(api_key=api_key)


In [ ]:
chat_model = ChatGoogleGenerativeAI(google_api_key=GEMINI,
                                   model="gemini-2.5-flash-lite")

## Load Dokumen PDF

Sebelum membangun RAG, kita perlu dokumen sumber yang akan jadi "pengetahuan" model.

Kita download PDF exam objectives dari Pearson VUE menggunakan `curl`,
lalu membacanya dengan `PyPDFLoader` dari LangChain.

`PyPDFLoader` secara otomatis:
- Membaca setiap halaman PDF
- Mengekstrak teksnya
- Membungkusnya sebagai objek `Document` dengan metadata halaman

> `load_and_split()` = load dokumen sekaligus split per halaman.
> Hasilnya adalah list of `Document`, satu item per halaman PDF.


In [ ]:
# download pdf dengan curl
!curl -o  ai_pv.pdf https://www.pearsonvue.com/content/dam/VUE/vue/en/documents/clients/it-specialist/its-od-307-artificial-intel-pearson.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  259k  100  259k    0     0   709k      0 --:--:-- --:--:-- --:--:--  711k


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("ai_pv.pdf")
pages = loader.load_and_split()

Kita cek isi halaman pertama dan total halaman untuk memastikan PDF berhasil dibaca.

In [ ]:
pages[0].page_content

'Copyright © 2025 Pearson Education, Inc. or its affiliates(s). All rights reserved.\n1. AI Problem Definition\n1.1 Identify the problem you are trying to solve using AI (e.g., user \nsegmentation, improving customer service)\n• Identify the need that will be addressed\n• Find out what information comes in and what output is expected\n• Determine whether AI is called for\n• Consider upsides and downsides of AI in the situation\n• Define measurable success\n• Benchmark against domain or organization-specific risks to which the \nproject may be susceptible\n1.2 Classify the problem (e.g., regression, unsupervised learning) \n• Examine available data (labeled or unlabeled?) and the problem\n• Determine problem type (e.g., classification, regression, unsupervised, \nreinforcement)\n1.3 Identify the areas of expertise needed to solve the problem\n• Identify business expertise required\n• Identify the need for domain (subject-matter) expertise on the problem\n• Identify AI expertise needed\n

In [ ]:
len(pages)

5

# Chunking — Memotong Dokumen Jadi Bagian Kecil

## Mengapa perlu Chunking?

LLM punya batas token — kita tidak bisa mengirim seluruh isi dokumen sekaligus.
Selain itu, mengirim seluruh dokumen ke model itu tidak efisien dan mahal.

Dengan chunking, dokumen dipotong jadi potongan-potongan kecil (chunk).
Saat user bertanya, kita hanya ambil chunk yang **paling relevan** dengan pertanyaan tersebut —
bukan keseluruhan dokumen.

## NLTKTextSplitter

Berbeda dengan splitter berbasis karakter biasa, **NLTKTextSplitter** memotong teks
berdasarkan **batas kalimat** menggunakan Natural Language Toolkit (NLTK).
Ini menghasilkan chunk yang lebih natural karena tidak memotong di tengah kalimat.

**Parameter penting:**
- `chunk_size` — maksimal panjang karakter per chunk. Terlalu kecil = konteks kurang, terlalu besar = tidak efisien
- `chunk_overlap` — berapa karakter dari akhir chunk sebelumnya yang diulang di awal chunk berikutnya,
  agar informasi di batas potongan tidak hilang

> NLTK perlu mendownload tokenizer `punkt_tab` terlebih dahulu sebelum bisa digunakan.


In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### Demo Chunking dengan Teks Sederhana

Sebelum diterapkan ke dokumen PDF, kita coba dulu dengan teks pendek
agar lebih mudah melihat bagaimana chunking bekerja.

In [ ]:
from langchain_text_splitters import NLTKTextSplitter

# Misalkan kamu memiliki dokumen seperti berikut
simple_doc = """halo nama saya sardi irfansyah, saya lahir di jakarta. Saya irfan. tinggal di Jakarta."""
print('panjang total karakter:',len(simple_doc),'\n')
# Membuat objek NLTKTextSplitter dengan ukuran chunk dan overlap
text_splitter = NLTKTextSplitter(separator='\n\n',chunk_size=67, chunk_overlap=10) #default separator='\n\n'

# Memecah dokumen menjadi beberapa chunk
chunks = text_splitter.split_text(simple_doc)
print(chunks,'\n')

# Menampilkan hasil chunk
for i, chunk in enumerate(chunks):
    #panjang karakter
    print(f"Panjang chunk {i+1}: {len(chunk)} karakter")
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 50)


panjang total karakter: 86 

['halo nama saya sardi irfansyah, saya lahir di jakarta.\n\nSaya irfan.', 'tinggal di Jakarta.'] 

Panjang chunk 1: 67 karakter
Chunk 1:
halo nama saya sardi irfansyah, saya lahir di jakarta.

Saya irfan.
--------------------------------------------------
Panjang chunk 2: 19 karakter
Chunk 2:
tinggal di Jakarta.
--------------------------------------------------


Penjelasan:
- Dapat kita lihat bahwa `NLTKTextSplitter` akan mencoba membuat text tersebut dipisahkan berdasarkan kalimat atau tanda `titik`. Jadi setiap ada titik maka akan dibuat chunk.
- Ketika panjang karakter lebih dari `chunk_size`, ini akan mengakibatkan peringatan warning.
- `chunk_overlap` digunakan untuk menentukan jumlah karakter yang harus tumpang tindih antara chunk yang berdekatan.

Jika anda ingin melihat ilustrasi tentang konfigurasi chunk, anda dapat melihatnya [di sini](https://dev.to/peterabel/what-chunk-size-and-chunk-overlap-should-you-use-4338).

### Terapkan Chunking ke Dokumen PDF

Sekarang kita terapkan splitter yang sama ke dokumen Pearson VUE yang sudah kita load.
Kita pakai `chunk_size=500` dan `chunk_overlap=100` sebagai trade-off antara
konteks yang cukup dan efisiensi pencarian.

In [ ]:
from langchain_text_splitters import NLTKTextSplitter

text_splitter = NLTKTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(pages)

print(len(chunks))

print(type(chunks[0]))

14
<class 'langchain_core.documents.base.Document'>


In [ ]:
# Mengecek hasil pemecahan menjadi chunk
print(f"Jumlah chunks yang dihasilkan: {len(chunks)}")
print(f"Tipe data chunk pertama: {type(chunks[0])}")

Jumlah chunks yang dihasilkan: 14
Tipe data chunk pertama: <class 'langchain_core.documents.base.Document'>


Kita lihat isi chunk pertama untuk memverifikasi hasil potongan.

In [ ]:
# Menampilkan chunk pertama
# panjang chunk
print(f"Panjang chunk pertama: {len(chunks[0].page_content)} karakter")
print("\nContoh chunk pertama:")
print(chunks[0].page_content)  # Memastikan bahwa setiap chunk memiliki konten

Panjang chunk pertama: 88 karakter

Contoh chunk pertama:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

1.


# Embedding — Mengubah Teks Menjadi Vector

## Apa itu Embedding?

Embedding adalah proses mengubah teks menjadi **deretan angka (vector)**
yang merepresentasikan makna dari teks tersebut.

```
"Apa skill yang dibutuhkan untuk sertifikasi AI?"  →  [0.12, -0.34, 0.87, ...]  (768 angka)
"Keterampilan apa yang perlu dikuasai untuk ujian AI?"  →  [0.11, -0.32, 0.85, ...]  (mirip!)
"Harga saham Tokopedia hari ini"  →  [-0.55, 0.22, -0.14, ...]  (jauh berbeda)
```

Teks yang maknanya **mirip** akan menghasilkan vector yang **berdekatan** di ruang vektor.
Ini yang memungkinkan pencarian semantik — mencari berdasarkan makna, bukan hanya kata kunci.

## Model Embedding yang Digunakan

`gemini-embedding-2` adalah model embedding dari Google yang menghasilkan vector berdimensi 768.
Kita set `output_dimensionality=768` untuk menyesuaikan dimensi dengan konfigurasi vector store.

> Proses embedding ini akan dijalankan untuk **setiap chunk** saat menyimpan ke ChromaDB,
> dan juga untuk **setiap pertanyaan** user saat melakukan pencarian.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(google_api_key=api_key, model="gemini-embedding-2", output_dimensionality=768)

# Vector Database — ChromaDB

## Apa itu Vector Database?

Vector database adalah database yang dirancang khusus untuk menyimpan dan mencari vector secara efisien.
Berbeda dengan database biasa yang mencari berdasarkan nilai eksak, vector database mencari
berdasarkan **kemiripan** (similarity) antar vector.

## ChromaDB

**ChromaDB** adalah vector database open-source yang bisa berjalan langsung di memori
atau disimpan ke disk. Di notebook ini kita pakai mode `persist_directory` agar
data tidak hilang setelah runtime Colab restart.

### Alur penyimpanan:

```
chunks (list of Document)
        ↓
  embedding_model.embed()    ← setiap chunk diubah jadi vector
        ↓
   ChromaDB.from_documents() ← vector + teks disimpan di ChromaDB
        ↓
   persist_directory="./chroma_db_"  ← disimpan ke disk
```

> `Chroma.from_documents()` melakukan dua hal sekaligus: embedding dan penyimpanan.
> Kita tidak perlu memanggil embed secara manual.

Dokumentasi vector store: [python.langchain.com/docs/how_to/vectorstore_retriever](https://python.langchain.com/docs/how_to/vectorstore_retriever/)


In [ ]:
from langchain_community.vectorstores import Chroma

# Sematkan setiap chunk dan muat ke dalam chromadb
db = Chroma.from_documents(chunks, embedding_model, persist_directory="./chroma_db_")

# Menyimpan perubahan ke disk
db.persist()

### Membuka Koneksi ke ChromaDB yang Sudah Tersimpan

Setelah data disimpan, kita buka koneksi baru ke database yang sama.
Ini memisahkan proses **penulisan** (from_documents) dan **pembacaan** (koneksi retriever),
yang penting untuk skenario production di mana embedding hanya dilakukan sekali.

In [ ]:
# mengatur koneksi untuk menghubungkan ke ChromaDB
db_connection = Chroma(persist_directory="./chroma_db_", embedding_function=embedding_model)

### Membuat Retriever

**Retriever** adalah objek yang tugasnya satu: menerima pertanyaan dan mengembalikan
dokumen-dokumen yang paling relevan dari vector database.

Di balik layar, retriever:
1. Mengubah pertanyaan user menjadi vector (embedding)
2. Mencari vector yang paling mirip di ChromaDB (similarity search)
3. Mengembalikan dokumen-dokumen dengan similarity tertinggi

In [ ]:
# Mengonversi koneksi Chroma menjadi objek retriever untuk pencarian dokumen berbasis vektor
retriever = db_connection.as_retriever(search_kwargs={"k": 10})

print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


`k` adalah parameter yang digunakan dalam fungsi as_retriever() untuk menentukan jumlah hasil pencarian teratas (top-k) yang akan diambil dari basis data vektor (Chroma database).

Parameter ini digunakan dalam proses pencarian dokumen relevan yang mengacu pada jumlah dokumen atau chunk yang akan diambil dari kumpulan dokumen yang lebih besar (misalnya, dari Chroma vector store) berdasarkan kesamaan atau kedekatannya dengan pertanyaan yang diajukan.

> Nilai k umumnya lebih kecil dari total chunk.



### Uji Coba Retriever

Sebelum membangun RAG chain, kita test dulu retriever-nya secara langsung.
Kita kirim pertanyaan dan lihat dokumen apa yang dikembalikan.

In [ ]:
check_response = retriever.invoke("jelaskan skill yang dibutuhkan untuk sertifikasi pearson vue dalam bidang AI?")
len(check_response)

10

Tampilkan isi chunk pertama yang dikembalikan retriever:

In [ ]:
md(check_response[0].page_content)

Successful candidates will be able to analyze and classify a problem.

They should 
be able to demonstrate knowledge of data collection, data processing, and feature engineering strategies.

Candidates should be able to choose an appropriate algorithm for training a model, and understand the 
metrics used to evaluate model performance.

They should understand the AI development lifecycle and 
how a production pipeline is used to allow for continuous improvement.

# Prompt Template

## Mengapa Perlu Template?

Kita tidak bisa langsung mengirim pertanyaan user ke LLM begitu saja.
LLM perlu **konteks** (dokumen yang sudah diambil retriever) beserta **instruksi** yang jelas.

Prompt template memungkinkan kita mendefinisikan struktur pesan secara konsisten,
dengan placeholder `{context}` dan `{question}` yang diisi secara otomatis saat runtime.

## Komponen Template di Notebook Ini

**SystemMessage** — instruksi tetap yang diberikan ke model di awal, mendefinisikan perannya:
```
"Anda adalah AI yang dapat menjawab pertanyaan berdasarkan konteks..."
```

**HumanMessagePromptTemplate** — template pesan dari user, berisi placeholder:
```
Jawab pertanyaan berikut berdasarkan konteks.
konteks: {context}         ← diisi dengan chunk dari ChromaDB
pertanyaan: {question}     ← diisi dengan pertanyaan user
jawaban:
```

Dokumentasi: [python.langchain.com/docs/concepts/prompt_templates](https://python.langchain.com/docs/concepts/prompt_templates/)


In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

In [ ]:
# Membuat template pesan untuk sistem dan pesan pengguna
chat_template = ChatPromptTemplate.from_messages([
    # System Message Prompt Template
    SystemMessage(content="""Anda adalah AI yang dapat menjawab pertanyaan berdasarkan konteks dan pertanyaan dari user.
                 Anda harus menjawab pertanyaan user, berdasarkan konteks"""),

    # Human Message Prompt Template
    HumanMessagePromptTemplate.from_template("""Jawab pertanyaan berikut berdasarkan konteks.
    konteks: {context}
    pertanyaan: {question}
    jawaban: """)
])

### Output Parser & Format Docs

**StrOutputParser** mengubah output LLM (yang berupa objek `AIMessage`) menjadi string biasa
agar lebih mudah ditampilkan.

**`format_docs`** adalah fungsi helper yang menggabungkan semua chunk yang dikembalikan retriever
menjadi satu string panjang, dipisahkan dua baris kosong. String inilah yang dimasukkan
ke placeholder `{context}` di prompt template.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

Kita lihat dulu seperti apa output dari `format_docs` sebelum dipakai di chain:

In [ ]:
# Setelah dokumen diambil oleh retriever dan di-format
formatted_docs = format_docs(chunks)  # Format dokumen yang dihasilkan dari text chunks

# Cetak hasil format
print("Hasil Format Docs:")
print(formatted_docs)  # Menampilkan hasil setelah dokumen diformat

Hasil Format Docs:
Copyright © 2025 Pearson Education, Inc. or its affiliates(s).

All rights reserved.

1.

AI Problem Definition
1.1 Identify the problem you are trying to solve using AI (e.g., user 
segmentation, improving customer service)
• Identify the need that will be addressed
• Find out what information comes in and what output is expected
• Determine whether AI is called for
• Consider upsides and downsides of AI in the situation
• Define measurable success
• Benchmark against domain or organization-specific risks to which the 
project may be susceptible
1.2 Classify the problem (e.g., regression, unsupervised learning) 
• Examine available data (labeled or unlabeled?)

and the problem
• Determine problem type (e.g., classification, regression, unsupervised, 
reinforcement)
1.3 Identify the areas of expertise needed to solve the problem
• Identify business expertise required
• Identify the need for domain (subject-matter) expertise on the problem
• Identify AI expertise need

## RAG Chain — Menggabungkan Semua Komponen

Inilah bagian utama dari notebook ini. Kita rangkai semua komponen menggunakan
**LCEL (LangChain Expression Language)** dengan operator pipe `|`.

```
{context: retriever | format_docs,   ← ambil dokumen relevan → format jadi string
 question: RunnablePassthrough()}     ← pertanyaan user diteruskan apa adanya
        |
  chat_template                       ← isi placeholder {context} dan {question}
        |
   chat_model                         ← kirim ke Gemini, dapat respons
        |
  output_parser                       ← ubah AIMessage → string biasa
```

**`RunnablePassthrough()`** berarti input (pertanyaan user) diteruskan langsung
tanpa modifikasi ke tahap berikutnya.

Seluruh chain ini bisa dijalankan cukup dengan `.invoke("pertanyaan kamu")`.


In [ ]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | chat_template
    | chat_model
    | output_parser
)


### Jalankan RAG Chain

Kita test dengan pertanyaan tentang sertifikasi Pearson VUE AI.
Chain akan otomatis: retrieve chunk relevan → format → kirim ke Gemini → return jawaban.

In [ ]:
response = rag_chain.invoke("""beritahu saya pemahaman apa yang dibutuhkan untuk sertifikasi IT spesialis: AI dari pearson vue""")

response

'Untuk mendapatkan sertifikasi IT Spesialis: AI dari Pearson Vue, Anda memerlukan pemahaman mendalam tentang:\n\n*   **Prosedur Pengembangan Solusi AI:** Termasuk cara mengembangkan solusi kecerdasan buatan.\n*   **Tata Kelola, Transparansi, Keamanan, dan Etika AI:** Memahami isu-isu seputar pengelolaan, keterbukaan, perlindungan, dan prinsip-prinsip etis dalam penggunaan AI.\n*   **Analisis dan Klasifikasi Masalah:** Mampu menganalisis dan mengklasifikasikan masalah yang akan diselesaikan dengan AI.\n*   **Pengumpulan, Pemrosesan, dan Rekayasa Data:** Memiliki pengetahuan tentang strategi pengumpulan data, pemrosesan data, dan rekayasa fitur.\n*   **Pemilihan Algoritma dan Evaluasi Model:** Mampu memilih algoritma yang tepat untuk melatih model dan memahami metrik yang digunakan untuk mengevaluasi kinerja model.\n*   **Siklus Hidup Pengembangan AI dan Pipeline Produksi:** Memahami siklus hidup pengembangan AI dan bagaimana pipeline produksi digunakan untuk perbaikan berkelanjutan.\n* 

Tampilkan jawaban dalam format Markdown agar lebih mudah dibaca:

In [ ]:
md(response)

Untuk mendapatkan sertifikasi IT Spesialis: AI dari Pearson Vue, Anda memerlukan pemahaman mendalam tentang:

*   **Prosedur Pengembangan Solusi AI:** Termasuk cara mengembangkan solusi kecerdasan buatan.
*   **Tata Kelola, Transparansi, Keamanan, dan Etika AI:** Memahami isu-isu seputar pengelolaan, keterbukaan, perlindungan, dan prinsip-prinsip etis dalam penggunaan AI.
*   **Analisis dan Klasifikasi Masalah:** Mampu menganalisis dan mengklasifikasikan masalah yang akan diselesaikan dengan AI.
*   **Pengumpulan, Pemrosesan, dan Rekayasa Data:** Memiliki pengetahuan tentang strategi pengumpulan data, pemrosesan data, dan rekayasa fitur.
*   **Pemilihan Algoritma dan Evaluasi Model:** Mampu memilih algoritma yang tepat untuk melatih model dan memahami metrik yang digunakan untuk mengevaluasi kinerja model.
*   **Siklus Hidup Pengembangan AI dan Pipeline Produksi:** Memahami siklus hidup pengembangan AI dan bagaimana pipeline produksi digunakan untuk perbaikan berkelanjutan.
*   **Algoritma dan Model AI:** Mengetahui cara mempertimbangkan penerapan algoritma spesifik dan memilih algoritma yang cocok (misalnya, jaringan saraf, klasifikasi seperti pohon keputusan, k-means).
*   **Pelatihan Model:** Mampu melatih model menggunakan algoritma yang dipilih dengan parameter awal yang diperkirakan.
*   **Pengukuran Dampak dan Penanganan Umpan Balik Pengguna:** Mampu mengukur dampak AI pada individu dan komunitas, serta menangani umpan balik dari pengguna.
*   **Definisi Masalah AI:** Mampu mengidentifikasi masalah yang ingin diselesaikan dengan AI, menentukan input dan output yang diharapkan, serta mendefinisikan keberhasilan yang terukur.
*   **Klasifikasi Masalah AI:** Mampu mengklasifikasikan masalah AI (misalnya, regresi, pembelajaran tanpa pengawasan).
*   **Pemilihan Cara Pengumpulan Data:** Menentukan jenis data yang dibutuhkan, apakah menggunakan dataset yang ada atau membuat sendiri, dan apakah pengumpulan dapat diotomatisasi atau memerlukan input pengguna.
*   **Penilaian Kualitas Data:** Menentukan apakah dataset memenuhi kebutuhan tugas, mencari data yang hilang atau rusak.
*   **Memastikan Representasi Data:** Memeriksa teknik pengumpulan untuk potensi bias dan memastikan jumlah data yang cukup untuk membangun model yang tidak bias.
*   **Identifikasi Kebutuhan Sumber Daya:** Menilai apakah masalah dapat diselesaikan dengan sumber daya komputasi yang tersedia dan mempertimbangkan anggaran proyek.
*   **Konversi Data:** Mengubah data ke format yang sesuai (misalnya, numerik, gambar, deret waktu).
*   **Pemilihan Fitur:** Menentukan fitur data yang akan dimasukkan dan membuat vektor fitur awal.
*   **Rekayasa Fitur:** Meninjau fitur dan melakukan transformasi standar untuk membuat dataset yang diproses.
*   **Identifikasi Dataset Pelatihan dan Pengujian:** Memisahkan data menjadi set pelatihan dan pengujian, serta memastikan set pengujian representatif.
*   **Dokumentasi Keputusan Data:** Mencatat asumsi, predikat, dan kendala dalam pilihan desain.
*   **Pemeliharaan dan Pemantauan AI dalam Produksi:** Melakukan pengawasan terhadap kinerja aplikasi dan model, menggunakan sistem pemantauan yang kuat, bertindak atas peringatan, dan mendeteksi kegagalan sistem.
*   **Penilaian Dampak Bisnis:** Melacak metrik kinerja utama untuk menentukan apakah solusi telah memecahkan masalah.
*   **Pemilihan Aktivitas Transparansi dan Validasi:** Mengkomunikasikan tujuan pengumpulan data, memutuskan siapa yang harus melihat hasil, dan meninjau persyaratan hukum.

Selain itu, kandidat diharapkan memiliki minimal 150 jam instruksi dan/atau eksplorasi metodologi dan solusi kecerdasan buatan.

# Contoh Lain: RAG dengan RecursiveCharacterTextSplitter & PromptTemplate

Sejauh ini kita sudah membangun RAG dengan `NLTKTextSplitter` dan `ChatPromptTemplate`.

Sekarang kita lihat **pendekatan alternatif** menggunakan:
- **`RecursiveCharacterTextSplitter`** — splitter berbasis karakter yang lebih umum dipakai
  dan tidak memerlukan NLTK. Memotong secara rekursif dengan berbagai separator (`

`, `
`, ` `)
  dari yang paling besar ke yang paling kecil sampai chunk cukup kecil.
- **`PromptTemplate`** + **`load_qa_chain`** — cara lama (legacy) LangChain untuk membangun QA chain
  sebelum LCEL diperkenalkan. Ditampilkan di sini sebagai perbandingan.

### Perbandingan dengan Pendekatan Sebelumnya

| Aspek | Pendekatan 1 (LCEL) | Pendekatan 2 (Legacy) |
|---|---|---|
| Splitter | NLTKTextSplitter (per kalimat) | RecursiveCharacterTextSplitter (per karakter) |
| Chain | LCEL pipe `\|` | `load_qa_chain` |
| Prompt | `ChatPromptTemplate` | `PromptTemplate` |
| Input PDF | via PyPDFLoader | via PyPDF2 langsung |
| Fleksibilitas | Lebih tinggi, modular | Lebih sederhana untuk pemula |

> **Catatan**: Pendekatan 2 menggunakan style lama LangChain yang sudah deprecated.
> Untuk project baru, gunakan pendekatan LCEL (Pendekatan 1).


In [ ]:
# Instalasi dependensi yang dibutuhkan
!pip install -q langchain PyPDF2 #python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.2 MB/s eta 0:00:00


### Import Library

Di sini kita import `PyPDF2` untuk membaca PDF secara manual (tanpa LangChain loader),
dan komponen-komponen legacy LangChain untuk membangun chain.

In [ ]:
import os
import io
import PyPDF2
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
# from dotenv import load_dotenv
from google.colab import files
from IPython.display import Markdown as md

### Baca PDF dan Ekstrak Teks

Kita baca file PDF yang sama (`ai_pv.pdf`) menggunakan PyPDF2.
Semua halaman digabungkan menjadi satu string `context`.

In [ ]:
# Membuka dan membaca file PDF
with open('/content/ai_pv.pdf', "rb") as file:
    pdf_reader = PyPDF2.PdfReader(file)
    pdf_pages = pdf_reader.pages

    # Mengekstrak teks dari semua halaman
    context = "\n\n".join(page.extract_text() for page in pdf_pages)

Cek isi teks yang sudah diekstrak:

In [ ]:
context

'Copyright © 2025 Pearson Education, Inc. or its affiliates(s). All rights reserved.\n1. AI Problem Definition\n1.1 Identify the problem you are trying to solve using AI (e.g., user \nsegmentation, improving customer service)\n• Identify the need that will be addressed\n• Find out what information comes in and what output is expected\n• Determine whether AI is called for\n• Consider upsides and downsides of AI in the situation\n• Define measurable success\n• Benchmark against domain or organization-specific risks to which the \nproject may be susceptible\n1.2 Classify the problem (e.g., regression, unsupervised learning) \n• Examine available data (labeled or unlabeled?) and the problem\n• Determine problem type (e.g., classification, regression, unsupervised, \nreinforcement)\n1.3 Identify the areas of expertise needed to solve the problem\n• Identify business expertise required\n• Identify the need for domain (subject-matter) expertise on the problem\n• Identify AI expertise needed\n

### Chunking dengan RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` memotong teks secara rekursif menggunakan beberapa separator
secara berurutan: `\n\n` → `\n` → ` ` → karakter individual.
Ini memastikan potongan selalu bermakna dan tidak terpotong di tengah kata.

In [ ]:
# Memecah teks menjadi potongan-potongan kecil
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_text(context)

Cek jumlah chunks yang dihasilkan dan isi chunk pertama:

In [ ]:
print(len(texts))

print(type(texts[0]))

24
<class 'str'>


In [ ]:
texts[0]

'Copyright © 2025 Pearson Education, Inc. or its affiliates(s). All rights reserved.\n1. AI Problem Definition\n1.1 Identify the problem you are trying to solve using AI (e.g., user \nsegmentation, improving customer service)\n• Identify the need that will be addressed\n• Find out what information comes in and what output is expected\n• Determine whether AI is called for\n• Consider upsides and downsides of AI in the situation\n• Define measurable success'

### Buat Embedding dan Vector Index

Di sini kita langsung membuat ChromaDB in-memory (tanpa `persist_directory`)
dan sekaligus mengonversinya menjadi retriever — semuanya dalam satu baris.

In [ ]:
# Membuat embeddings untuk potongan-potongan teks
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2",google_api_key=api_key)

vector_index = Chroma.from_texts(texts, embeddings).as_retriever(search_kwargs={"k": 20})

Cek objek retriever yang dihasilkan:

In [ ]:
vector_index

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7a4c349dd4f0>, search_kwargs={'k': 20})

In [ ]:
print(type(vector_index))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


### Tanya Jawab Interaktif

Di sini user bisa mengetik pertanyaan secara langsung via `input()`.
Retriever akan mencari dokumen relevan, lalu chain akan menghasilkan jawaban.

In [ ]:
# Mendapatkan pertanyaan dari pengguna
user_question = input("Tanyakan pertanyaan: ")

# Mendapatkan dokumen relevan untuk pertanyaan pengguna
docs = vector_index.invoke(user_question)

Tanyakan pertanyaan: Apa isi dokumen ini?


### Buat Prompt, Chain, dan Generate Jawaban

Kita definisikan template prompt, buat chain menggunakan `load_qa_chain` (style lama),
lalu panggil dengan dokumen yang sudah diambil retriever.

In [ ]:
# Mendefinisikan template prompt
prompt_template = """
Jawablah pertanyaan ini dengan se-detail mungkin dari konteks yang diberikan,
pastikan untuk memberikan semua detail, jika jawaban tidak ada dalam
konteks yang diberikan cukup katakan, "jawaban tidak tersedia dalam konteks",
jangan memberikan jawaban yang salah\n\n
Konteks:\n {context}?\n
Pertanyaan: \n{question}\n
Jawaban:
"""

# Membuat prompt
prompt = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])

# Memuat QA chain
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", api_key=api_key)
chain = load_qa_chain(model, chain_type="stuff", prompt=prompt)

# Mendapatkan jawaban dari model
response = chain({"input_documents": docs, "question": user_question}, return_only_outputs=True)

# Menampilkan jawaban
print("\nJawaban:")
md(response['output_text'])

/tmp/ipykernel_8371/2582920223.py:17: LangChainDeprecationWarning: The function `load_qa_chain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  chain = load_qa_chain(model, chain_type="stuff", prompt=prompt)
/tmp/ipykernel_8371/2582920223.py:20: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = chain({"input_documents": docs, "question": user_question}, return_only_outputs=True)



Jawaban:


Dokumen ini berisi "IT SPECIALIST EXAM OBJECTIVES" yang berkaitan dengan Kecerdasan Buatan (AI). Dokumen ini menguraikan pengetahuan dan keterampilan yang diharapkan dari kandidat sertifikasi IT Specialist, khususnya dalam bidang AI.

Secara rinci, dokumen ini mencakup beberapa area utama:

**1. Analisis dan Klasifikasi Masalah:**
*   Memahami prosedur pengembangan solusi AI.
*   Menganalisis dan mengklasifikasikan masalah (misalnya, regresi, pembelajaran tanpa pengawasan).
*   Mengidentifikasi area keahlian yang dibutuhkan untuk memecahkan masalah.
*   Memastikan AI digunakan secara tepat, termasuk mengidentifikasi potensi kesalahan prediksi atau kerugian pada kelompok pengguna tertentu, serta mempertimbangkan penggunaan hasil AI di luar konteks.

**2. Pengumpulan dan Pemrosesan Data:**
*   Memilih cara mengumpulkan data, termasuk menentukan jenis/karakteristik data yang dibutuhkan dan apakah perlu membuat dataset sendiri.
*   Menilai kualitas data, termasuk memeriksa kelengkapan dan kebenaran data.
*   Memastikan data representatif dengan memeriksa teknik pengumpulan untuk potensi bias dan memastikan jumlah data yang cukup untuk model yang tidak bias.
*   Mengidentifikasi kebutuhan sumber daya (komputasi, kompleksitas waktu, anggaran).
*   Mengonversi data ke format yang sesuai (misalnya, numerik, gambar, deret waktu) dan ke dalam fitur yang cocok untuk AI.
*   Memilih fitur untuk model AI dan melakukan rekayasa fitur (membuat dataset yang diproses).
*   Memisahkan data menjadi dataset pelatihan dan pengujian, serta memastikan dataset pengujian representatif.
*   Mendokumentasikan keputusan terkait data, termasuk asumsi, predikat, dan batasan.

**3. Algoritma dan Model AI:**
*   Mempertimbangkan penerapan algoritma spesifik (misalnya, jaringan saraf, klasifikasi seperti decision tree, k-means).
*   Melatih model menggunakan algoritma yang dipilih dan menyetel model dengan mengubah parameter.
*   Mengevaluasi kinerja model (misalnya, akurasi, presisi), memeriksa overfitting/underfitting, menghasilkan metrik/KPI, dan melakukan validasi silang dengan data uji baru.
*   Mencari potensi bias dalam algoritma, memastikan masukan menyerupai data pelatihan, dan mengonfirmasi bahwa data pelatihan tidak mengandung korelasi yang tidak relevan.
*   Mengevaluasi sensitivitas model, termasuk menguji spesifisitas model.
*   Mengonfirmasi kepatuhan terhadap persyaratan peraturan, jika ada, dengan mengevaluasi keluaran sesuai ambang batas yang ditentukan dan mendokumentasikan hasil.
*   Memperoleh persetujuan pemangku kepentingan dengan mengumpulkan hasil, membandingkan risiko, dan mengadakan sesi evaluasi solusi.

**4. Integrasi dan Penerapan Aplikasi:**
*   Melatih pelanggan tentang cara menggunakan produk dan apa yang diharapkan dari produk tersebut, termasuk menginformasikan batasan model, penggunaan yang dimaksudkan, berbagi dokumentasi, dan mengelola ekspektasi pelanggan.
*   Merencanakan penanganan tantangan potensial model dalam produksi, termasuk memahami jenis tantangan yang mungkin dihadapi, indikator tantangan, dan cara mitigasinya.
*   Merancang pipeline produksi, termasuk integrasi aplikasi, untuk memenuhi kebutuhan produk.
*   Mendukung solusi AI dengan mendokumentasikan fungsi untuk pemeliharaan, melatih tim pendukung, mengimplementasikan mekanisme umpan balik, detektor drift, dan cara mengumpulkan data baru.

**5. Pemeliharaan dan Pemantauan AI dalam Produksi:**
*   Melakukan pengawasan dengan mencatat kinerja aplikasi dan model untuk keamanan, debugging, akuntabilitas, dan audit, menggunakan sistem pemantauan yang kuat, dan bertindak berdasarkan peringatan.
*   Mengamati sistem dari waktu ke waktu untuk memeriksa drift atau penurunan kinerja, dan mendeteksi kegagalan sistem dalam mendukung informasi baru.
*   Menilai dampak bisnis (KPI) dengan melacak metrik untuk menentukan apakah solusi telah memecahkan masalah dan bertindak berdasarkan metrik yang tidak terduga.
*   Mengukur dampak pada individu dan komunitas, termasuk menganalisis dampak pada subkelompok tertentu dan mengidentifikasi peluang optimasi.
*   Menangani umpan balik dari pengguna dengan mengukur kepuasan pengguna, menilai kebingungan pengguna, dan memasukkan umpan balik ke dalam versi mendatang.
*   Mempertimbangkan peningkatan atau penghentian secara berkala dengan menggabungkan observasi dampak untuk menilai nilai AI, dan memutuskan apakah akan melatih ulang AI, terus menggunakan AI sebagaimana adanya, atau menghentikan AI.

Secara keseluruhan, dokumen ini memberikan kerangka kerja komprehensif untuk pengembangan, penerapan, dan pemeliharaan solusi AI, dengan penekanan kuat pada kepatuhan, transparansi, keamanan, dan etika.